# Visual Assistant - Complete Fixed Notebook

**Run order: Cell 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> then Cell 11**

- Fill in your Gemini API key in Cell 3
- Enter your ngrok token in Cell 7 when prompted

Get free Gemini key: https://aistudio.google.com

Get free ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken

## Cell 1 - Install All Dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q openai-whisper
!pip install -q gtts pydub
!pip install -q ultralytics
!pip install -q Pillow opencv-python-headless
!pip install -q sentencepiece protobuf soundfile
!pip install -q google-generativeai
!pip install -q flask flask-cors
!pip install -q pyngrok
!apt-get install -q ffmpeg
print('All packages installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 39.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
All packages installed!


## Cell 2 - Imports

In [ ]:
import os, io, base64, json, time, tempfile, warnings, re, threading
warnings.filterwarnings('ignore')

import torch
import numpy as np
import cv2
import requests as req_lib
from PIL import Image, ImageEnhance, ImageDraw
import whisper
from gtts import gTTS
from ultralytics import YOLO
from flask import Flask, request, jsonify
from flask_cors import CORS

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## Cell 3 - Gemini API Key

Replace `YOUR_GEMINI_API_KEY_HERE` with your key from https://aistudio.google.com

If you get rate limit errors, create a new key with a different Google account.

In [ ]:
GEMINI_API_KEY = 'Your API KEY'  # keep your current key - it works!
GEMINI_MODEL = 'gemini-3-flash-preview'

print('Testing Gemini API key...')

def _test_gemini_key(key, model):
    url = ('https://generativelanguage.googleapis.com/v1beta/models/'
           + model + ':generateContent?key=' + key)
    payload = {
        'contents': [{'parts': [{'text': 'Say OK'}]}],
        'generationConfig': {'maxOutputTokens': 50}  # fixed: was 10, too small
    }
    r = req_lib.post(url, json=payload, timeout=20)
    if r.status_code != 200:
        return False, f'HTTP {r.status_code}: {r.text[:300]}'
    data = r.json()
    try:
        candidates = data.get('candidates', [])
        if not candidates:
            return False, 'No candidates: ' + str(data)
        content = candidates[0].get('content', {})
        parts = content.get('parts', [])
        if not parts:
            finish = candidates[0].get('finishReason', 'UNKNOWN')
            return False, 'No parts. finishReason: ' + finish
        return True, parts[0].get('text', 'No text')
    except Exception as e:
        return False, 'Parse error: ' + str(e)

ok, msg = _test_gemini_key(GEMINI_API_KEY, GEMINI_MODEL)
if ok:
    print('Gemini API key works! Model: ' + GEMINI_MODEL)
    print('Response: ' + msg)
else:
    print('Gemini key failed: ' + msg)

Testing Gemini API key...
Gemini key failed: No parts. finishReason: MAX_TOKENS


## Cell 4 - Load Whisper and YOLO

In [ ]:
print('Loading Whisper (medium)...')
whisper_model = whisper.load_model('medium')
print('Whisper loaded!')

print('Loading YOLOv8...')
yolo_model = YOLO('yolov8m.pt')
print('YOLO loaded!')

print('All models ready!')


Loading Whisper (medium)...


100%|█████████████████████████████████████| 1.42G/1.42G [00:34<00:00, 43.9MiB/s]


Whisper loaded!
Loading YOLOv8...
YOLO loaded!
All models ready!


## Cell 5 - Helper Functions

Gemini uses direct REST API to bypass the broken Colab proxy.

In [ ]:
def preprocess_image(pil_image):
    img = pil_image.convert('RGB')
    img_array = np.array(img)
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
    is_blurry = blur_score < 100
    if is_blurry:
        print(f'  [Blurry image (score={blur_score:.1f}), enhancing...]')
        blurred = cv2.GaussianBlur(img_array, (0, 0), 3)
        img_array = cv2.addWeighted(img_array, 1.5, blurred, -0.5, 0)
    enhanced = Image.fromarray(np.clip(img_array, 0, 255).astype(np.uint8))
    enhanced = ImageEnhance.Contrast(enhanced).enhance(1.3)
    w, h = enhanced.size
    if w < 336 or h < 336:
        scale = max(336/w, 336/h)
        enhanced = enhanced.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
    return enhanced, is_blurry


def run_yolo_detection(pil_image):
    results = yolo_model(pil_image, verbose=False, conf=0.35)
    detections = {}
    for result in results:
        for box in result.boxes:
            cls_name = yolo_model.names[int(box.cls)]
            detections[cls_name] = detections.get(cls_name, 0) + 1
    if not detections:
        return ''
    parts = []
    for obj, count in sorted(detections.items(), key=lambda x: -x[1]):
        parts.append(f'{count} {obj}' if count > 1 else obj)
    return 'Detected objects: ' + ', '.join(parts) + '.'


def run_gemini(pil_image, prompt):
    # Direct REST API - bypasses broken Kaggle/Colab SDK proxy at localhost:42287
    for attempt in range(3):
        try:
            buf = io.BytesIO()
            pil_image.convert('RGB').save(buf, format='JPEG', quality=85)
            img_b64 = base64.b64encode(buf.getvalue()).decode()

            url = ('https://generativelanguage.googleapis.com/v1beta/models/'
                   + GEMINI_MODEL + ':generateContent?key=' + GEMINI_API_KEY)
            payload = {
                'contents': [{
                    'parts': [
                        {'inline_data': {'mime_type': 'image/jpeg', 'data': img_b64}},
                        {'text': prompt}
                    ]
                }],
                'generationConfig': {'temperature': 0.1, 'maxOutputTokens': 1024}
            }

            r = req_lib.post(url, json=payload, timeout=60)

            if r.status_code == 429:
                wait = 30
                match = re.search(r'"retryDelay"\s*:\s*"(\d+)', r.text)
                if match:
                    wait = int(match.group(1)) + 5
                wait = min(wait, 65)
                print(f'  Rate limit. Retrying in {wait}s...')
                time.sleep(wait)
                continue
            if r.status_code == 403:
                print(f'  API key invalid: {r.text[:200]}')
                return 'API key error. Please check your Gemini API key.'
            if r.status_code != 200:
                print(f'  HTTP {r.status_code}: {r.text[:200]}')
                time.sleep(5)
                continue

            data = r.json()
            text = data['candidates'][0]['content']['parts'][0]['text']
            return text.strip()

        except req_lib.Timeout:
            print(f'  Timeout attempt {attempt+1}/3, retrying...')
            time.sleep(5)
        except Exception as e:
            print(f'  Attempt {attempt+1}/3 error: {e}')
            time.sleep(3)

    return 'I was unable to analyze this image at the moment.'


def text_to_audio(text, output_path=None):
    if output_path is None:
        output_path = tempfile.mktemp(suffix='.mp3')
    tts = gTTS(text=text, lang='en', slow=False)
    tts.save(output_path)
    return output_path


def audio_to_text(audio_path):
    result = whisper_model.transcribe(
        audio_path,
        language='en',
        fp16=torch.cuda.is_available()
    )
    return result['text'].strip()


print('Helper functions defined!')


Helper functions defined!


## Cell 6 - VisualAssistant Class

In [ ]:
from IPython.display import display as ipy_display

class VisualAssistant:
    def __init__(self):
        self.current_image = None
        self.conversation_history = []
        self.image_metadata = {}

    def load_image_from_url(self, url):
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = req_lib.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content)).convert('RGB')

    def analyze_image(self, pil_image):
        print('Analyzing image...')
        display_img = pil_image.copy()
        display_img.thumbnail((400, 400))
        ipy_display(display_img)

        enhanced_image, was_blurry = preprocess_image(pil_image)
        self.current_image = enhanced_image
        self.conversation_history = []

        print('  Running object detection...')
        yolo_info = run_yolo_detection(enhanced_image)

        grounding = ''
        if yolo_info:
            grounding += '\nDetected objects: ' + yolo_info
        if was_blurry:
            grounding += '\nNote: Image appears blurry.'

        is_document = len(yolo_info) == 0

        if is_document:
            prompt = ('You are a precise assistant helping a visually impaired person read an image.'
                + grounding
                + '\n\nThis image appears to be a document or text-based image.'
                + '\nRead and transcribe ALL the text completely and accurately.'
                + '\nDo not summarize - read every word from top to bottom.'
                + '\nAfter the full text, add one sentence describing the document type.')
        else:
            prompt = ('You are a precise assistant helping a visually impaired person understand an image.'
                + grounding
                + '\n\nDescribe this image in detail. Cover:'
                + '\n- Main subject: what is it, what is happening'
                + '\n- Visual details: colors, shapes, size, texture'
                + '\n- Layout and background'
                + '\n- Any text visible: read it exactly'
                + '\n- Any people: age, expression, clothing, action'
                + '\n- Overall mood or context'
                + '\n\nRules:'
                + '\n- Write 4-6 complete sentences'
                + '\n- Be specific and factual'
                + '\n- Do NOT say "I can see" - just describe directly'
                + '\n- Do NOT invent details not visible')

        print('  Running Gemini description...')
        description = run_gemini(enhanced_image, prompt)

        self.image_metadata = {
            'yolo': yolo_info,
            'was_blurry': was_blurry,
            'description': description
        }

        print('  Generating audio...')
        audio_path = text_to_audio(description)

        print('Analysis complete!')
        print('\nDescription:\n' + description)
        return description, audio_path

    def answer_question(self, question_text):
        time.sleep(1)
        if self.current_image is None:
            msg = 'No image analyzed yet. Please describe an image first.'
            return msg, text_to_audio(msg)

        print('\nQuestion: ' + question_text)

        history_str = ''
        for turn in self.conversation_history[-4:]:
            history_str += 'User: ' + turn['q'] + '\nAssistant: ' + turn['a'] + '\n'

        grounding = ''
        if self.image_metadata.get('yolo'):
            grounding += '\n' + self.image_metadata['yolo']

        prompt = ('You are a precise assistant helping a visually impaired person.'
            + '\n\nImage description: ' + self.image_metadata.get('description', '')
            + '\n' + grounding
            + '\n\nConversation so far:\n' + history_str
            + '\nAnswer based ONLY on what is visible in the image.'
            + '\nBe factual and concise - 1 to 2 sentences.'
            + '\nDo NOT guess or add details not in the image.'
            + '\n\nQuestion: ' + question_text
            + '\nAnswer:')

        answer = run_gemini(self.current_image, prompt)
        self.conversation_history.append({'q': question_text, 'a': answer})
        audio_path = text_to_audio(answer)
        print('Answer: ' + answer)
        return answer, audio_path

    def answer_voice_question(self, audio_path):
        print('Transcribing...')
        question = audio_to_text(audio_path)
        print('Transcribed: "' + question + '"')
        if not question or len(question.strip()) < 2:
            msg = 'I could not understand your question. Please try again.'
            return '', msg, text_to_audio(msg)
        answer_text, audio_out = self.answer_question(question)
        return question, answer_text, audio_out


assistant = VisualAssistant()
print('VisualAssistant ready!')


VisualAssistant ready!


## Cell 7 - Flask Server and ngrok Tunnel

Get your free ngrok token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import subprocess, threading, time
import requests as req
from pyngrok import ngrok, conf
import getpass

subprocess.run(['fuser', '-k', '5000/tcp'], capture_output=True)
time.sleep(2)

app = Flask(__name__)
CORS(app)

@app.after_request
def add_headers(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = 'Content-Type, ngrok-skip-browser-warning'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return response

@app.route('/health', methods=['GET', 'OPTIONS'])
def health():
    if request.method == 'OPTIONS':
        return '', 204
    return jsonify({'status': 'ok', 'gpu': torch.cuda.is_available()})

@app.route('/analyze_image', methods=['POST', 'OPTIONS'])
def analyze_image_endpoint():
    if request.method == 'OPTIONS':
        return '', 204
    try:
        data = request.get_json()
        if 'image_base64' in data and data['image_base64']:
            img_bytes = base64.b64decode(data['image_base64'])
            img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        elif 'image_url' in data and data['image_url']:
            img = assistant.load_image_from_url(data['image_url'])
        else:
            return jsonify({'error': 'No image provided'}), 400
        description, audio_path = assistant.analyze_image(img)
        with open(audio_path, 'rb') as f:
            audio_b64 = base64.b64encode(f.read()).decode('utf-8')
        return jsonify({'description': description, 'audio_base64': audio_b64})
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/ask_voice', methods=['POST', 'OPTIONS'])
def ask_voice_endpoint():
    if request.method == 'OPTIONS':
        return '', 204
    try:
        if assistant.current_image is None:
            msg = 'No image analyzed yet. Please describe an image first.'
            audio_path = text_to_audio(msg)
            with open(audio_path, 'rb') as f:
                audio_b64 = base64.b64encode(f.read()).decode('utf-8')
            return jsonify({'question': '', 'answer': msg, 'audio_base64': audio_b64})
        data = request.get_json()
        audio_bytes = base64.b64decode(data['audio_base64'])
        audio_path = '/tmp/voice_q.webm'
        with open(audio_path, 'wb') as f:
            f.write(audio_bytes)
        question, answer, audio_out = assistant.answer_voice_question(audio_path)
        with open(audio_out, 'rb') as f:
            audio_b64 = base64.b64encode(f.read()).decode('utf-8')
        return jsonify({'question': question, 'answer': answer, 'audio_base64': audio_b64})
    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@app.route('/ask_text', methods=['POST', 'OPTIONS'])
def ask_text_endpoint():
    if request.method == 'OPTIONS':
        return '', 204
    try:
        data = request.get_json()
        question = data.get('question', '')
        answer, audio_path = assistant.answer_question(question)
        with open(audio_path, 'rb') as f:
            audio_b64 = base64.b64encode(f.read()).decode('utf-8')
        return jsonify({'answer': answer, 'audio_base64': audio_b64})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)

try:
    r = req.get('http://localhost:5000/health', timeout=5)
    print('Flask running:', r.json())
except Exception as e:
    print('Flask failed:', e)

print('\nGet your free token at: https://dashboard.ngrok.com/get-started/your-authtoken')
NGROK_TOKEN = getpass.getpass('Paste your ngrok authtoken and press Enter: ')

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
time.sleep(2)

try:
    tunnel = ngrok.connect(5000, bind_tls=True)
    PUBLIC_URL = tunnel.public_url
    print('\n' + '=' * 60)
    print('Server is live!')
    print('YOUR URL: ' + PUBLIC_URL)
    print('=' * 60)
    print('Paste this URL into your Chrome extension popup!\n')

    time.sleep(3)
    r2 = req.get(PUBLIC_URL + '/health', headers={'ngrok-skip-browser-warning': 'true'}, timeout=15)
    print('Tunnel test passed:', r2.json())
    print('\nAll good! Now run Cell 11 to download the extension.')
except Exception as e:
    print('ngrok failed: ' + str(e))
    print('Check your authtoken and try again.')


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [15/Mar/2026 14:45:09] "GET /health HTTP/1.1" 200 -


Flask running: {'gpu': True, 'status': 'ok'}

Get your free token at: https://dashboard.ngrok.com/get-started/your-authtoken
Paste your ngrok authtoken and press Enter: ··········

Server is live!
YOUR URL: https://swirly-autosuggestible-lelah.ngrok-free.dev
Paste this URL into your Chrome extension popup!



INFO:werkzeug:127.0.0.1 - - [15/Mar/2026 14:46:41] "GET /health HTTP/1.1" 200 -


Tunnel test passed: {'gpu': True, 'status': 'ok'}

All good! Now run Cell 11 to download the extension.


## Cell 8 - Test Upload (Optional)

Upload an image to verify the full pipeline works.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

print('Upload an image to test...')
uploaded = files.upload()

for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert('RGB')
    print('\nProcessing: ' + fname)
    description, audio_path = assistant.analyze_image(img)
    print('\nPlaying description audio:')
    display(Audio(audio_path, autoplay=True))
    break


Upload an image to test...


TypeError: 'NoneType' object is not subscriptable

## Cell 9 - Voice Q&A

Run Cell 8 first to load an image.

In [ ]:
from IPython.display import display, Audio, HTML
from google.colab import output

def process_voice_answer(audio_b64):
    if audio_b64 == 'END_SESSION':
        print('Session ended. Total questions: ' + str(len(assistant.conversation_history)))
        for i, turn in enumerate(assistant.conversation_history, 1):
            print('  Q' + str(i) + ': ' + turn['q'])
            print('  A' + str(i) + ': ' + turn['a'])
        return
    try:
        audio_bytes = base64.b64decode(audio_b64)
        audio_path = '/tmp/voice_question.webm'
        with open(audio_path, 'wb') as f:
            f.write(audio_bytes)
        print('Transcribing...')
        question = audio_to_text(audio_path)
        print('You asked: "' + question + '"')
        if not question or len(question.strip()) < 2:
            print('Could not understand. Please try again.')
            return
        answer, audio_out = assistant.answer_question(question)
        display(Audio(audio_out, autoplay=True))
        print('Total questions: ' + str(len(assistant.conversation_history)))
    except Exception as e:
        print('Error: ' + str(e))
        import traceback
        traceback.print_exc()

output.register_callback('process_voice_answer', process_voice_answer)

def start_voice_qa():
    if assistant.current_image is None:
        print('Please run Cell 8 first!')
        return
    print('Voice Q&A Mode')
    display(HTML('''
    <style>
      .va-btn{padding:12px 24px;margin:6px;border:none;border-radius:8px;font-size:15px;cursor:pointer;font-weight:bold}
      #startBtn{background:#4CAF50;color:white}#stopBtn{background:#f44336;color:white}#endBtn{background:#555;color:white}
      #va-status{margin-top:12px;padding:10px 14px;border-radius:8px;font-size:14px;font-family:Arial,sans-serif;background:#1a1a2e;color:#eee;min-height:36px}
    </style>
    <div>
      <button class="va-btn" id="startBtn">Start Recording</button>
      <button class="va-btn" id="stopBtn" disabled>Stop Recording</button>
      <button class="va-btn" id="endBtn">End Session</button>
      <div id="va-status">Ready - click Start Recording.</div>
    </div>
    <script>
      let mediaRecorder=null,audioChunks=[];
      function setStatus(msg){document.getElementById("va-status").innerText=msg;}
      document.getElementById("startBtn").addEventListener("click",async()=>{
        try{
          const stream=await navigator.mediaDevices.getUserMedia({audio:true});
          audioChunks=[];mediaRecorder=new MediaRecorder(stream);
          mediaRecorder.ondataavailable=e=>{if(e.data.size>0)audioChunks.push(e.data);};
          mediaRecorder.onstop=async()=>{
            setStatus("Processing...");
            const blob=new Blob(audioChunks,{type:"audio/webm"});
            const reader=new FileReader();
            reader.onloadend=()=>{google.colab.kernel.invokeFunction("process_voice_answer",[reader.result.split(",")[1]],{});};
            reader.readAsDataURL(blob);stream.getTracks().forEach(t=>t.stop());
          };
          mediaRecorder.start();
          document.getElementById("startBtn").disabled=true;
          document.getElementById("stopBtn").disabled=false;
          setStatus("Recording... speak now.");
        }catch(err){setStatus("Mic denied: "+err.message);}
      });
      document.getElementById("stopBtn").addEventListener("click",()=>{
        if(mediaRecorder&&mediaRecorder.state!=="inactive"){
          mediaRecorder.stop();
          document.getElementById("startBtn").disabled=false;
          document.getElementById("stopBtn").disabled=true;
          setStatus("Sending to AI...");
        }
      });
      document.getElementById("endBtn").addEventListener("click",()=>{
        setStatus("Session ended.");
        document.getElementById("startBtn").disabled=true;
        document.getElementById("stopBtn").disabled=true;
        google.colab.kernel.invokeFunction("process_voice_answer",["END_SESSION"],{});
      });
    </script>
    '''))

start_voice_qa()


## Cell 10 - Text Q&A Fallback

Use this if you have no microphone.

In [ ]:
from IPython.display import Audio, display

def text_qa_loop():
    if assistant.current_image is None:
        print('Please run Cell 8 first!')
        return
    print('\nText Q&A Mode - type stop to end\n')
    while True:
        question = input('Your question: ').strip()
        if not question:
            continue
        if question.lower() == 'stop':
            print('Session ended.')
            break
        answer, audio_path = assistant.answer_question(question)
        display(Audio(audio_path, autoplay=True))
        print()

text_qa_loop()


## Cell 11 - Generate Chrome Extension ZIP

Run this after Cell 7. Downloads the extension with all fixes built in.

In [ ]:
import os, json, shutil
from PIL import Image, ImageDraw
from google.colab import files

EXT_DIR = '/content/nvda_extension'
if os.path.exists(EXT_DIR):
    shutil.rmtree(EXT_DIR)
os.makedirs(EXT_DIR, exist_ok=True)

print('Building extension for URL: ' + PUBLIC_URL)

# manifest.json
with open(EXT_DIR + '/manifest.json', 'w') as f:
    json.dump({
        'manifest_version': 3,
        'name': 'Visual Assistant for NVDA',
        'version': '7.0.0',
        'description': 'AI image description and voice Q&A for visually impaired users',
        'permissions': ['activeTab', 'scripting', 'storage', 'contextMenus'],
        'host_permissions': ['<all_urls>'],
        'background': {'service_worker': 'background.js'},
        'content_scripts': [{'matches': ['<all_urls>'], 'js': ['content.js'], 'run_at': 'document_end'}],
        'action': {'default_popup': 'popup.html', 'default_title': 'Visual Assistant'}
    }, f, indent=2)

# background.js
with open(EXT_DIR + '/background.js', 'w') as f:
    f.write(
"let API_URL = '';\n"
"chrome.storage.sync.get(['apiUrl'], r => { API_URL = r.apiUrl || ''; });\n"
"chrome.storage.onChanged.addListener(c => { if (c.apiUrl) API_URL = c.apiUrl.newValue; });\n"
"\n"
"chrome.runtime.onInstalled.addListener(() => {\n"
"  chrome.contextMenus.create({ id: 'describeImage', title: 'Describe this image (Visual Assistant)', contexts: ['image'] });\n"
"});\n"
"\n"
"chrome.contextMenus.onClicked.addListener((info, tab) => {\n"
"  if (info.menuItemId === 'describeImage')\n"
"    chrome.tabs.sendMessage(tab.id, { action: 'analyzeImageUrl', imageUrl: info.srcUrl, apiUrl: API_URL });\n"
"});\n"
"\n"
"const HEADERS = { 'Content-Type': 'application/json', 'ngrok-skip-browser-warning': 'true' };\n"
"\n"
"chrome.runtime.onConnect.addListener(port => {\n"
"  port.onMessage.addListener(async msg => {\n"
"\n"
"    if (msg.action === 'fetchHealth') {\n"
"      const url = (msg.apiUrl || API_URL).replace(/\\/$/,'');\n"
"      try {\n"
"        const res = await fetch(url+'/health',{method:'GET',headers:HEADERS,signal:AbortSignal.timeout(10000)});\n"
"        const ct = res.headers.get('content-type')||'';\n"
"        if (!ct.includes('application/json')) { port.postMessage({ok:false,error:'Non-JSON: '+(await res.text()).substring(0,100)}); return; }\n"
"        port.postMessage({ok:true,data:await res.json()});\n"
"      } catch(e) { port.postMessage({ok:false,error:e.message}); }\n"
"    }\n"
"\n"
"    if (msg.action === 'fetchAnalyzeUrl') {\n"
"      const url = (msg.apiUrl || API_URL).replace(/\\/$/,'');\n"
"      try {\n"
"        const hb = setInterval(()=>{ try{port.postMessage({heartbeat:true});}catch(e){clearInterval(hb);} },5000);\n"
"        const res = await fetch(url+'/analyze_image',{method:'POST',headers:HEADERS,body:JSON.stringify({image_url:msg.imageUrl})});\n"
"        clearInterval(hb);\n"
"        const ct = res.headers.get('content-type')||'';\n"
"        if (!ct.includes('application/json')) { port.postMessage({ok:false,error:'Non-JSON: '+(await res.text()).substring(0,100)}); return; }\n"
"        port.postMessage({ok:true,data:await res.json()});\n"
"      } catch(e) { port.postMessage({ok:false,error:e.message}); }\n"
"    }\n"
"\n"
"    if (msg.action === 'fetchAskVoice') {\n"
"      const url = (msg.apiUrl || API_URL).replace(/\\/$/,'');\n"
"      try {\n"
"        const hb = setInterval(()=>{ try{port.postMessage({heartbeat:true});}catch(e){clearInterval(hb);} },5000);\n"
"        const res = await fetch(url+'/ask_voice',{method:'POST',headers:HEADERS,body:JSON.stringify({audio_base64:msg.audioBase64})});\n"
"        clearInterval(hb);\n"
"        const ct = res.headers.get('content-type')||'';\n"
"        if (!ct.includes('application/json')) { port.postMessage({ok:false,error:'Non-JSON: '+(await res.text()).substring(0,100)}); return; }\n"
"        port.postMessage({ok:true,data:await res.json()});\n"
"      } catch(e) { port.postMessage({ok:false,error:e.message}); }\n"
"    }\n"
"\n"
"  });\n"
"});\n"
    )

# content.js
with open(EXT_DIR + '/content.js', 'w') as f:
    f.write(
"let currentAudio=null,mediaRecorder=null,audioChunks=[],isRecording=false,apiBaseUrl='';\n"
"chrome.storage.sync.get(['apiUrl'],r=>{apiBaseUrl=r.apiUrl||'';});\n"
"chrome.storage.onChanged.addListener(c=>{if(c.apiUrl)apiBaseUrl=c.apiUrl.newValue;});\n"
"\n"
"function announceToNVDA(text){\n"
"  let r=document.getElementById('va-nvda-live');\n"
"  if(!r){r=document.createElement('div');r.id='va-nvda-live';\n"
"    r.setAttribute('aria-live','assertive');r.setAttribute('aria-atomic','true');r.setAttribute('role','status');\n"
"    r.style.cssText='position:absolute;left:-9999px;width:1px;height:1px;overflow:hidden;';\n"
"    document.body.appendChild(r);}\n"
"  r.textContent='';setTimeout(()=>{r.textContent=text;},50);\n"
"}\n"
"\n"
"function playAudioBase64(b64){\n"
"  try{\n"
"    if(currentAudio){currentAudio.pause();currentAudio=null;}\n"
"    const bytes=atob(b64),arr=new Uint8Array(bytes.length);\n"
"    for(let i=0;i<bytes.length;i++)arr[i]=bytes.charCodeAt(i);\n"
"    currentAudio=new Audio(URL.createObjectURL(new Blob([arr],{type:'audio/mp3'})));\n"
"    currentAudio.play().catch(e=>console.error('[VA]',e));\n"
"  }catch(e){console.error('[VA] playAudio:',e);}\n"
"}\n"
"\n"
"function updatePanel(text,state){\n"
"  const el=document.getElementById('va-status');if(!el)return;\n"
"  el.style.background=({ready:'#0d0d1a',processing:'#1a2a00',recording:'#2a0000',error:'#3a0000'})[state]||'#0d0d1a';\n"
"  el.textContent=text;\n"
"}\n"
"\n"
"function sendViaPort(message,onResult){\n"
"  const port=chrome.runtime.connect({name:'va-request'});\n"
"  let settled=false;\n"
"  port.onMessage.addListener(r=>{\n"
"    if(r.heartbeat)return;\n"
"    if(!settled){settled=true;port.disconnect();onResult(r);}\n"
"  });\n"
"  port.onDisconnect.addListener(()=>{if(!settled){settled=true;onResult({ok:false,error:'Background disconnected.'});}});\n"
"  port.postMessage(message);\n"
"}\n"
"\n"
"function analyzeImage(imageUrl){\n"
"  if(!apiBaseUrl){announceToNVDA('Set API URL in popup first.');updatePanel('Set API URL in popup first.','error');return;}\n"
"  announceToNVDA('Analyzing image, please wait about 30 seconds.');\n"
"  updatePanel('Analyzing image... Please wait 30-60 seconds.','processing');\n"
"  sendViaPort({action:'fetchAnalyzeUrl',imageUrl,apiUrl:apiBaseUrl},r=>{\n"
"    if(!r.ok){announceToNVDA('Error: '+r.error);updatePanel('Error: '+r.error,'error');return;}\n"
"    if(r.data.error){announceToNVDA('Error: '+r.data.error);updatePanel('Error: '+r.data.error,'error');return;}\n"
"    announceToNVDA(r.data.description);updatePanel(r.data.description,'ready');playAudioBase64(r.data.audio_base64);\n"
"  });\n"
"}\n"
"\n"
"async function startVoiceRecording(){\n"
"  try{\n"
"    const stream=await navigator.mediaDevices.getUserMedia({audio:true});\n"
"    audioChunks=[];mediaRecorder=new MediaRecorder(stream);\n"
"    mediaRecorder.ondataavailable=e=>{if(e.data.size>0)audioChunks.push(e.data);};\n"
"    mediaRecorder.onstop=sendVoiceQuestion;mediaRecorder.start();isRecording=true;\n"
"    announceToNVDA('Recording. Press Alt Shift Q to stop.');\n"
"    updatePanel('Recording... press Alt+Shift+Q to stop','recording');\n"
"    const btn=document.getElementById('va-mic');if(btn)btn.textContent='Stop';\n"
"  }catch(e){announceToNVDA('Mic denied.');updatePanel('Mic denied.','error');}\n"
"}\n"
"\n"
"function stopVoiceRecording(){\n"
"  if(mediaRecorder&&isRecording){\n"
"    mediaRecorder.stop();mediaRecorder.stream.getTracks().forEach(t=>t.stop());isRecording=false;\n"
"    updatePanel('Processing... (30-60 seconds)','processing');\n"
"    const btn=document.getElementById('va-mic');if(btn)btn.textContent='Ask';\n"
"  }\n"
"}\n"
"\n"
"function sendVoiceQuestion(){\n"
"  const reader=new FileReader();\n"
"  reader.onloadend=()=>{\n"
"    sendViaPort({action:'fetchAskVoice',audioBase64:reader.result.split(',')[1],apiUrl:apiBaseUrl},r=>{\n"
"      if(!r.ok){announceToNVDA('Error: '+r.error);updatePanel('Error: '+r.error,'error');return;}\n"
"      if(r.data.error){announceToNVDA('Error: '+r.data.error);updatePanel('Error: '+r.data.error,'error');return;}\n"
"      announceToNVDA('You asked: '+r.data.question+'. Answer: '+r.data.answer);\n"
"      updatePanel('Q: '+r.data.question+'\\nA: '+r.data.answer,'ready');\n"
"      playAudioBase64(r.data.audio_base64);\n"
"    });\n"
"  };\n"
"  reader.readAsDataURL(new Blob(audioChunks,{type:'audio/webm'}));\n"
"}\n"
"\n"
"function createPanel(){\n"
"  if(document.getElementById('va-panel'))return;\n"
"  const panel=document.createElement('div');\n"
"  panel.id='va-panel';panel.setAttribute('role','dialog');panel.setAttribute('aria-label','Visual Assistant');\n"
"  panel.style.cssText='position:fixed;bottom:20px;right:20px;width:340px;background:#1a1a2e;color:#eee;border-radius:12px;padding:16px;z-index:999999;font-family:Arial,sans-serif;font-size:14px;box-shadow:0 4px 20px rgba(0,0,0,0.5);border:2px solid #4a90e2;';\n"
"  const hdr=document.createElement('div');hdr.style.cssText='display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;';\n"
"  const ttl=document.createElement('strong');ttl.style.color='#4a90e2';ttl.textContent='Visual Assistant';\n"
"  const cls=document.createElement('button');cls.setAttribute('aria-label','Close');\n"
"  cls.style.cssText='background:none;border:none;color:#eee;font-size:18px;cursor:pointer;';cls.textContent='X';\n"
"  cls.addEventListener('click',()=>panel.remove());hdr.appendChild(ttl);hdr.appendChild(cls);\n"
"  const st=document.createElement('div');st.id='va-status';st.setAttribute('aria-live','polite');\n"
"  st.style.cssText='min-height:60px;background:#0d0d1a;padding:10px;border-radius:8px;margin-bottom:10px;font-size:13px;word-wrap:break-word;white-space:pre-wrap;';\n"
"  st.textContent='Ready. Right-click any image to describe it.';\n"
"  const br=document.createElement('div');br.style.cssText='display:flex;gap:8px;';\n"
"  const mb=document.createElement('button');mb.id='va-mic';mb.setAttribute('aria-label','Ask voice question');\n"
"  mb.style.cssText='flex:1;padding:10px;background:#4a90e2;color:white;border:none;border-radius:8px;cursor:pointer;font-size:13px;';\n"
"  mb.textContent='Ask';mb.addEventListener('click',()=>{isRecording?stopVoiceRecording():startVoiceRecording();});\n"
"  const sb=document.createElement('button');sb.setAttribute('aria-label','Stop audio');\n"
"  sb.style.cssText='flex:1;padding:10px;background:#555;color:white;border:none;border-radius:8px;cursor:pointer;font-size:13px;';\n"
"  sb.textContent='Stop Audio';sb.addEventListener('click',()=>{if(currentAudio){currentAudio.pause();currentAudio=null;}});\n"
"  br.appendChild(mb);br.appendChild(sb);\n"
"  const hl=document.createElement('div');hl.style.cssText='font-size:11px;color:#888;margin-top:8px;';\n"
"  hl.innerHTML='Alt+Shift+D: Describe focused image<br>Alt+Shift+Q: Ask/stop voice<br>Enter on image: Describe it<br>Takes 30-60 seconds';\n"
"  panel.appendChild(hdr);panel.appendChild(st);panel.appendChild(br);panel.appendChild(hl);\n"
"  document.body.appendChild(panel);\n"
"}\n"
"\n"
"function enhanceImages(){\n"
"  document.querySelectorAll('img').forEach(img=>{\n"
"    if(img.dataset.vaProcessed)return;img.dataset.vaProcessed='1';\n"
"    if(!img.hasAttribute('tabindex'))img.setAttribute('tabindex','0');\n"
"    if(!img.title)img.title='Press Enter to describe with Visual Assistant';\n"
"    img.addEventListener('focus',()=>createPanel());\n"
"    img.addEventListener('keydown',e=>{if(e.key==='Enter'||e.key===' '){e.preventDefault();createPanel();analyzeImage(img.src);}});\n"
"  });\n"
"}\n"
"\n"
"document.addEventListener('keydown',e=>{\n"
"  if(e.altKey&&e.shiftKey&&e.key==='D'){const el=document.activeElement;if(el&&el.tagName==='IMG'){createPanel();analyzeImage(el.src);}}\n"
"  if(e.altKey&&e.shiftKey&&e.key==='Q'){createPanel();isRecording?stopVoiceRecording():startVoiceRecording();}\n"
"});\n"
"\n"
"enhanceImages();\n"
"new MutationObserver(enhanceImages).observe(document.body,{childList:true,subtree:true});\n"
"chrome.runtime.onMessage.addListener(msg=>{\n"
"  if(msg.action==='analyzeImageUrl'){apiBaseUrl=msg.apiUrl||apiBaseUrl;createPanel();analyzeImage(msg.imageUrl);}\n"
"});\n"
    )

# popup.html
with open(EXT_DIR + '/popup.html', 'w') as f:
    f.write('<!DOCTYPE html>\n<html lang="en">\n<head><meta charset="UTF-8"><title>Visual Assistant</title><link rel="stylesheet" href="popup.css"></head>\n<body>\n  <h2>Visual Assistant</h2>\n  <label for="apiUrl">ngrok Server URL:</label><br><br>\n  <input type="url" id="apiUrl" placeholder="https://xxxx.ngrok-free.app">\n  <button id="saveBtn">Save and Test Connection</button>\n  <div id="status" class="status"></div>\n  <div class="help">\n    <strong>How to use:</strong><br>\n    1. Right-click any image and select Describe this image<br>\n    2. Wait 30-60 seconds for description and audio<br>\n    3. Press Ask or Alt+Shift+Q for voice questions<br><br>\n    <strong>Shortcuts:</strong><br>\n    Alt+Shift+D: Describe focused image<br>\n    Alt+Shift+Q: Ask or stop voice<br>\n    Enter on image: Describe it\n  </div>\n  <script src="popup.js"></script>\n</body>\n</html>')

# popup.css
with open(EXT_DIR + '/popup.css', 'w') as f:
    f.write('body{font-family:Arial,sans-serif;width:300px;padding:16px;background:#1a1a2e;color:#eee;margin:0}\nh2{color:#4a90e2;margin:0 0 12px;font-size:16px}label{font-size:13px}\ninput{width:100%;padding:8px;border-radius:6px;border:1px solid #4a90e2;background:#0d0d1a;color:#eee;box-sizing:border-box;font-size:13px}\nbutton{margin-top:8px;width:100%;padding:10px;background:#4a90e2;color:white;border:none;border-radius:6px;cursor:pointer;font-size:14px}\nbutton:hover{background:#357abd}.status{margin-top:8px;padding:8px;border-radius:6px;font-size:12px;display:none}\n.ok{background:#1a3a1a;color:#5f5}.err{background:#3a1a1a;color:#f55}\n.help{font-size:11px;color:#aaa;margin-top:12px;line-height:1.7}')

# popup.js
with open(EXT_DIR + '/popup.js', 'w') as f:
    f.write(
"const input=document.getElementById('apiUrl'),statusEl=document.getElementById('status'),saveBtn=document.getElementById('saveBtn');\n"
"chrome.storage.sync.get(['apiUrl'],r=>{if(r.apiUrl)input.value=r.apiUrl;});\n"
"saveBtn.addEventListener('click',saveAndTest);\n"
"function saveAndTest(){\n"
"  const url=input.value.trim().replace(/\\/$/,'');\n"
"  if(!url){show('Please enter a URL',false);return;}\n"
"  chrome.storage.sync.set({apiUrl:url});show('Testing connection...',null);\n"
"  const port=chrome.runtime.connect({name:'va-request'});\n"
"  let settled=false;\n"
"  port.onMessage.addListener(r=>{\n"
"    if(r.heartbeat)return;\n"
"    if(!settled){settled=true;port.disconnect();\n"
"      if(!r.ok){show('Failed: '+(r.error||'Cannot connect.'),false);return;}\n"
"      show('Connected! GPU: '+(r.data.gpu?'Yes':'No'),true);}\n"
"  });\n"
"  port.onDisconnect.addListener(()=>{if(!settled){settled=true;show('Failed: disconnected.',false);}});\n"
"  port.postMessage({action:'fetchHealth',apiUrl:url});\n"
"}\n"
"function show(msg,ok){\n"
"  statusEl.style.display='block';\n"
"  statusEl.className='status '+(ok===true?'ok':ok===false?'err':'');\n"
"  statusEl.textContent=msg;\n"
"}\n"
    )

# Icons
for size in [16, 48, 128]:
    icon = Image.new('RGBA', (size, size), (0,0,0,0))
    draw = ImageDraw.Draw(icon)
    draw.ellipse([2,2,size-2,size-2], fill=(74,144,226,255))
    icon.save(EXT_DIR + '/icon' + str(size) + '.png')

if os.path.exists('/content/nvda_extension.zip'):
    os.remove('/content/nvda_extension.zip')

shutil.make_archive('/content/nvda_extension', 'zip', EXT_DIR)
print('Extension ZIP created!')
print('\nInstall Steps:')
print('1. Extract the downloaded ZIP')
print('2. Go to chrome://extensions and enable Developer Mode')
print('3. Remove any old Visual Assistant extension')
print('4. Click Load unpacked and select the extracted folder')
print('5. Paste this URL in the popup: ' + PUBLIC_URL)
print('6. Click Save and Test - should show Connected')
print('7. Right-click any image and select Describe this image')
print('   Wait 30-60 seconds for the result')

files.download('/content/nvda_extension.zip')
